In [ ]:
# %pip install astroquery firefly_client

import http.client
import time
import urllib.error

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

from astroquery.ipac.irsa import Irsa

from firefly_client import FireflyClient

# The time it takes to read SPHEREx files can exceed
# astropy's default timeout limit. Increase it.
from astropy.utils.data import conf
conf.remote_timeout = 120

In [36]:
ra_deg = 304.693508808
dec_deg = 42.4436872991

coord = SkyCoord(ra_deg, dec_deg, unit='deg')
search_radius = 1 * u.arcsec

Irsa.list_collections(servicetype='SIA', filter='spherex')

results = Irsa.query_sia(pos=(coord, search_radius), collection=['spherex_dr1', 'spherex_qr3'])

spectral_image_url = results['access_url'][0]

print(spectral_image_url)
print(len(results))
print(results.colnames)

https://irsa.ipac.caltech.edu/ibe/data/spherex/dr1/level2/2025W18_1B/l2b-v27-2026-196/1/level2_2025W18_1B_0074_2D1_spx_l2b-v27-2026-196.fits
443
['s_ra', 's_dec', 'facility_name', 'instrument_name', 'dataproduct_subtype', 'calib_level', 'dataproduct_type', 'energy_bandpassname', 'energy_emband', 'obs_id', 's_resolution', 'em_min', 'em_max', 'em_res_power', 'proposal_title', 'access_url', 'access_format', 'access_estsize', 't_exptime', 's_region', 'obs_collection', 'obs_intent', 'algorithm_name', 'facility_keywords', 'instrument_keywords', 'environment_photometric', 'proposal_id', 'proposal_pi', 'proposal_project', 'target_name', 'target_type', 'target_standard', 'target_moving', 'target_keywords', 'obs_release_date', 's_xel1', 's_xel2', 's_pixel_scale', 'position_timedependent', 't_min', 't_max', 't_resolution', 't_xel', 'obs_publisher_did', 's_fov', 'em_xel', 'pol_states', 'pol_xel', 'cloud_access', 'o_ucd', 'upload_row_id']


In [37]:
# Max number of times to retry HTTP errors.
# max_retries = 3
# for attempt in range(max_retries):
#     try:
#         # Load the data, reading every extension into memory so that
#         # hdulist stays usable after the file is closed.
#         with fits.open(spectral_image_url, memmap=False) as hdulist:
#             for hdu in hdulist:
#                 hdu.data
#         break
#     except (TimeoutError, urllib.error.HTTPError, http.client.IncompleteRead):
#         if attempt == max_retries - 1:
#             raise
#         time.sleep(10 * (attempt + 1))


hdu = fits.open("./data/level2_2025W18_1B_0074_2D1_spx_l2b-v27-2026-196.fits")

hdu.info()

data = hdu["IMAGE"].data

wcs = WCS(hdu["IMAGE"].header)

ra_deg = wcs.pixel_to_world(100, 200)
print(f"RA: {ra_deg.ra.deg}, Dec: {ra_deg.dec.deg}")


Filename: ./data/level2_2025W18_1B_0074_2D1_spx_l2b-v27-2026-196.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  IMAGE         1 ImageHDU       293   (2040, 2040)   float32   
  2  FLAGS         1 CompImageHDU    185   (2040, 2040)   int32   
  3  VARIANCE      1 ImageHDU       119   (2040, 2040)   float32   
  4  ZODI          1 ImageHDU       122   (2040, 2040)   float32   
  5  EPSF          1 BinTableHDU     56   441R x 11C   [J, J, E, E, E, E, J, E, E, E, 1089D]   
  6  WCS-WAVE      1 BinTableHDU     22   1R x 3C   [9J, 9J, 162E]   
RA: 306.33851741190534, Dec: 39.71262416331999


In [30]:
fc = FireflyClient.make_client(url="https://irsa.ipac.caltech.edu/irsaviewer")

fc.show_fits_image(file_input=spectral_image_url, title="SPHEREx Spectral Image",
                plot_id="spectral_image")

fc.reinit_viewer()

{'success': True}

In [31]:
spectral_image_header = hdu[1].header

keywords_to_print = ['EXTNAME', 'NAXIS1', 'NAXIS2', 'BUNIT', 'DETECTOR', 'OBSID', 'DATE', 'PSF_FWHM']

for keyword in keywords_to_print:
    value = spectral_image_header.get(keyword, 'Keyword not found')
    print(f"{keyword}: {value}")

EXTNAME: IMAGE
NAXIS1: 2040
NAXIS2: 2040
BUNIT: MJy / sr
DETECTOR: 1
OBSID: 2025W18_1B_0074_2
DATE: 2026-07-16T03:19:15.597
PSF_FWHM: 7.530770327317724


In [35]:
wcs

ra, dec = wcs.wcs.crval

x, y = wcs.world_to_pixel(SkyCoord(ra, dec, unit="deg"))

print(f"Pixel coordinates of the reference point: x={x}, y={y}")

Pixel coordinates of the reference point: x=1019.0880637763953, y=1020.746373505631


In [40]:
spectral_image_header['WCSNAME*']

ra, dec = wcs.wcs.crval


x, y = wcs.world_to_pixel(SkyCoord(ra, dec, unit="deg"))

print(f"RA: {ra}, Dec: {dec}    Pixel coordinates of the reference point: x={x}, y={y}")

spectral_image_header['WCSNAME*']

RA: 303.877766184, Dec: 40.5489585415    Pixel coordinates of the reference point: x=1019.0880637763953, y=1020.746373505631


WCSNAMEA= '0-based active pixel coordinates' / Coordinate system title          
WCSNAMEW= 'Approximated per-pixel wavelength and bandwidth' / WCS label         

In [43]:
spectral_wcs = WCS(
    header=hdu["IMAGE"].header,
    fobj=hdu,
    key="W"
)

spectral_wcs.sip = None  

spectral_wcs = WCS(
    header=hdu["IMAGE"].header,
    fobj=hdu,
    key="W"
)

spectral_wcs.sip = None

print(spectral_wcs)

INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]
INFO: 
             

In [44]:
x = 1019.0880637763953
y = 1020.746373505631

# Obtener coordenadas espectrales del píxel
wl, bw = spectral_wcs.pixel_to_world(x, y)

print("Longitud de onda:", wl)
print("Ancho de banda:", bw)

Longitud de onda: 0.9041668563939969 um
Ancho de banda: 0.022984030955561494 um
